Install Libraries

In [1]:
!pip install datasets audiomentations -q
from datasets import load_dataset, DatasetDict, Dataset
from datasets.features import Audio
from sklearn.model_selection import train_test_split
import pandas as pd
import audiomentations as A
from tqdm import tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.1/86.1 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.5/248.5 kB 13.1 MB/s eta 0:00:00


Load Dataset

In [2]:
dataset = load_dataset("PolyAI/minds14", "en-US")
dataset

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

en-US/train-00000-of-00001.parquet:   0%|          | 0.00/34.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/563 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['path', 'audio', 'transcription', 'english_transcription', 'intent_class', 'lang_id'],
        num_rows: 563
    })
})

Cek Struktur Data

In [3]:
print("\n-------Struktur Data-----------")
print(f"Kolom: {dataset['train'].column_names}")
print(f"Jumlah Data Train Asli: {len(dataset['train'])}")
print(f"\n Contoh Data:")
print( dataset['train'][0])


-------Struktur Data-----------
Kolom: ['path', 'audio', 'transcription', 'english_transcription', 'intent_class', 'lang_id']
Jumlah Data Train Asli: 563

 Contoh Data:
{'path': 'en-US~JOINT_ACCOUNT/602ba55abb1e6d0fbce92065.wav', 'audio': <datasets.features._torchcodec.AudioDecoder object at 0x79fed8780470>, 'transcription': 'I would like to set up a joint account with my partner', 'english_transcription': 'I would like to set up a joint account with my partner', 'intent_class': 11, 'lang_id': 4}


Hapus Kolom yang tidak digunakan

In [4]:
columns_to_remove = ['lang_id', 'english_transcription', 'path']
dataset = dataset.remove_columns(columns_to_remove)
dataset

DatasetDict({
    train: Dataset({
        features: ['audio', 'transcription', 'intent_class'],
        num_rows: 563
    })
})

Split Data

In [5]:
full_data = dataset['train']
indices=list(range(len(full_data)))
train_idx, temp_idx = train_test_split(indices, test_size=0.2, random_state=42, stratify=full_data['intent_class'])
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=42, stratify=[full_data['intent_class'][i] for i in temp_idx])

#Buat Dataset Splits
train_data = full_data.select(train_idx)
val_data = full_data.select(val_idx)
test_data = full_data.select(test_idx)

In [6]:
#Gabungkan ke DatasetDict
dataset_split = DatasetDict({
    'train': train_data,
    'validation': val_data,
    'test': test_data
})

dataset_split

DatasetDict({
    train: Dataset({
        features: ['audio', 'transcription', 'intent_class'],
        num_rows: 450
    })
    validation: Dataset({
        features: ['audio', 'transcription', 'intent_class'],
        num_rows: 56
    })
    test: Dataset({
        features: ['audio', 'transcription', 'intent_class'],
        num_rows: 57
    })
})

Resample dan Augment data train

In [7]:
#Resample
train_data = dataset_split['train'].cast_column("audio", Audio(sampling_rate=16_000))

In [8]:
#Augment

#Definisi
augmenter = A.Compose([
    A.AddGaussianNoise(min_amplitude=0.001, max_amplitude=0.015, p=0.5),
    A.PitchShift(min_semitones=-2, max_semitones=2, p=0.5),
    A.TimeStretch(min_rate=0.8, max_rate=1.25, p=0.5),
    ])

In [9]:
#Augment

#Definisi
augmenter = A.Compose([
    A.AddGaussianNoise(min_amplitude=0.001, max_amplitude=0.015, p=0.5),
    A.PitchShift(min_semitones=-2, max_semitones=2, p=0.5),
    A.TimeStretch(min_rate=0.8, max_rate=1.25, p=0.5),
])

# Initialize empty lists to store augmented data
augmented_audios_data = [] # This will store dicts: {'array': ..., 'sampling_rate': ...}
augmented_intents = []
augmented_transcripts = []

for sample in tqdm(train_data):
  audio_array = sample['audio']['array']
  sr = sample['audio']['sampling_rate']
  intent = sample['intent_class']
  transcript = sample['transcription']

  # Add original sample
  augmented_audios_data.append({'array': audio_array, 'sampling_rate': sr})
  augmented_intents.append(intent)
  augmented_transcripts.append(transcript)

  # Add 3 augmented versions per original sample
  for _ in range(3):
    augmented_audio_array = augmenter(samples=audio_array, sample_rate=sr)
    augmented_audios_data.append({'array': augmented_audio_array, 'sampling_rate': sr})
    augmented_intents.append(intent)
    augmented_transcripts.append(transcript)

# Create the new dataset from the augmented data
train_data = Dataset.from_dict({
    'audio': augmented_audios_data,
    'intent_class': augmented_intents,
    'transcription': augmented_transcripts
}, features=train_data.features)

print(f"Selesai! Jumlah data train sebelum augmentasi: {len(train_data) // 4} -> setelah augmentasi: {len(train_data)}")

100%|██████████| 450/450 [01:35<00:00,  4.73it/s]


Selesai! Jumlah data train sebelum augmentasi: 450 -> setelah augmentasi: 1800


In [10]:
#Gabungkan ke DatasetDict
dataset_split_minds14 = DatasetDict({
    'train': train_data,
    'validation': val_data,
    'test': test_data
})

dataset_split_minds14

DatasetDict({
    train: Dataset({
        features: ['audio', 'intent_class', 'transcription'],
        num_rows: 1800
    })
    validation: Dataset({
        features: ['audio', 'transcription', 'intent_class'],
        num_rows: 56
    })
    test: Dataset({
        features: ['audio', 'transcription', 'intent_class'],
        num_rows: 57
    })
})

## Simpan Dataset

In [13]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [21]:
dataset_split.save_to_disk("/content/drive/MyDrive/Colab Notebooks/data project 3 (Voice Recognition)/data_nalapro_project03")
dataset_split_minds14.save_to_disk("/content/drive/MyDrive/Colab Notebooks/data project 3 (Voice Recognition)/data_nalapro_project03")

Saving the dataset (0/1 shards):   0%|          | 0/450 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/56 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/57 [00:00<?, ? examples/s]

Saving the dataset (0/2 shards):   0%|          | 0/1800 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/56 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/57 [00:00<?, ? examples/s]

In [22]:
#Zip folder
!zip -r /content/drive/MyDrive/Colab\ Notebooks/data\ project\ 3\ \(Voice\ Recognition\)/data_nalapro_project03.zip /content/drive/MyDrive/Colab\ Notebooks/data\ project\ 3\ \(Voice\ Recognition\)/data_nalapro_project03
print("Zip siap di: /content/drive/MyDrive/Colab Notebooks/data project 3 (Voice Recognition)/data_nalapro_project03.zip")

  adding: content/drive/MyDrive/Colab Notebooks/data project 3 (Voice Recognition)/data_nalapro_project03/ (stored 0%)
  adding: content/drive/MyDrive/Colab Notebooks/data project 3 (Voice Recognition)/data_nalapro_project03/dataset_dict.json (deflated 5%)
  adding: content/drive/MyDrive/Colab Notebooks/data project 3 (Voice Recognition)/data_nalapro_project03/train/ (stored 0%)
  adding: content/drive/MyDrive/Colab Notebooks/data project 3 (Voice Recognition)/data_nalapro_project03/train/data-00000-of-00001.arrow (deflated 24%)
  adding: content/drive/MyDrive/Colab Notebooks/data project 3 (Voice Recognition)/data_nalapro_project03/train/state.json (deflated 47%)
  adding: content/drive/MyDrive/Colab Notebooks/data project 3 (Voice Recognition)/data_nalapro_project03/train/dataset_info.json (deflated 57%)
  adding: content/drive/MyDrive/Colab Notebooks/data project 3 (Voice Recognition)/data_nalapro_project03/train/data-00000-of-00002.arrow (deflated 26%)
  adding: content/drive/MyDri

In [14]:
import zipfile
import os

In [15]:
#Folder Induk
os.makedirs("/kaggle/working/data_nalapro_project03", exist_ok=True)


In [20]:
#Salin kedua dataset ke folder induk
!cp -r /kaggle/working/dataset_split_minds14 /kaggle/working/data_nalapro_project03/
!cp -r /kaggle/working/dataset_split /kaggle/working/data_nalapro_project03/

cp: cannot stat '/kaggle/working/dataset_split_minds14': No such file or directory
cp: cannot stat '/kaggle/working/dataset_split': No such file or directory


In [17]:
#Zip folder induk
zip_path = "/kaggle/working/data_nalapro_project03.zip"
!zip -r {zip_path} /kaggle/working/data_nalapro_project03

print(f"Zip siap di: {zip_path}")

  adding: kaggle/working/data_nalapro_project03/ (stored 0%)
Zip siap di: /kaggle/working/data_nalapro_project03.zip


In [19]:
# Salin ke google Drive
!cp {zip_path} "/content/drive/MyDrive/Colab Notebooks/data project 3 (Voice Recognition)"